## MAPPINGS
- *Phase 1 (f1): Utgångsposition*
- *Phase 2 (f2): Rörelsestart*
- *Phase 3 (f3): Rörelseutförande*
- *Tider är relativa Utgångsposition (t = 0.0): `relativ_tid = abs_filmtid − Utgångsposition_tid`*

In [23]:
import pandas as pd
import os

In [24]:
OUTPUT_DIR = '../data/processed/OMA_Score'

FILES = {
    '../data/original/Forts_IRAF_SAT_Kors _Gemensamt.xlsx': f'{OUTPUT_DIR}/OMA_Score_Kors.csv',
    '../data/original/Final_IRAF_SAT_Rak_Bedömare sep och gemensamt.xlsx':  f'{OUTPUT_DIR}/OMA_Score_Rak.csv',
}

PHASE_MAPPING = {
    'Utgångsposition':  'f1',
    'Rörelsestart':     'f2',
    'Rörelseutförande': 'f3',
}

MOVEMENT_MAPPING = {
    'Sittande till stående': '2a_1',
    'Stående till sittande': '2a_2',
}

PERSON_MAPPING = {
    'sb': 'fp1',
    'cf': 'fp2',
    'jn': 'fp3',
    'al': 'fp4',
    'wl': 'fp5',
}

In [25]:
def ensure_output_directory(file_path):
    directory = os.path.dirname(file_path)
    if directory and not os.path.exists(directory):
        os.makedirs(directory)

def parse_time(raw_tid):
    if pd.isna(raw_tid):
        return "", ""
    tid_str = str(raw_tid).strip()
    if '-' in tid_str:
        parts = tid_str.split('-', 1)
        return parts[0].strip(), parts[1].strip()
    return tid_str, ""

def to_relative(abs_time, origin):
    try:
        t = float(str(abs_time).strip())
        o = float(str(origin).strip())
        return round(max(0.0, t - o), 2)
    except (TypeError, ValueError):
        return ""

In [26]:
def extract_oma_data(file_path, phase_mapping, movement_mapping, person_mapping):
    xl = pd.ExcelFile(file_path)
    all_extracted_rows = []

    for sheet_name in xl.sheet_names:
        movement_id = movement_mapping.get(sheet_name)
        if movement_id is None:
            continue

        df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

        current_person_id  = None
        current_phase_id   = None
        current_start_tid  = ''
        current_end_tid    = ''
        current_global_origin = None   
        current_aspekt     = 0
        extracted_rows     = []

        for _, row in df.iterrows():
            col0    = str(row.iloc[0]).replace('\xa0', ' ').strip() if pd.notna(row.iloc[0]) else ''
            col1    = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ''
            raw_tid = row.iloc[2] if pd.notna(row.iloc[2]) else None
            score   = row.iloc[12] if len(row) > 12 and pd.notna(row.iloc[12]) else None
            saker   = row.iloc[13] if len(row) > 13 and pd.notna(row.iloc[13]) else None

            if col1 == 'Viktning' and col0.startswith('Fp'):
                current_person_id = col0.split('_')[0].lower()
                current_phase_id = None; current_aspekt = 0; current_global_origin = None
                continue

            if not col0 or col0.startswith(('Film Nr:', 'Total tid')) or col1 == 'Viktning':
                continue

            if col0.startswith('Bedömare'):
                current_person_id = current_phase_id = None
                current_start_tid = current_end_tid = ''
                current_global_origin = None
                current_aspekt = 0
                continue

            if col0.startswith('Försöksperson:'):
                current_person_id = person_mapping.get(col0.split(':')[1].split()[0].lower().rstrip('.'))
                continue

            new_phase_id = phase_mapping.get(col0)
            if new_phase_id:
                if current_phase_id in ('f1', 'f2') and raw_tid is not None:
                    s, _ = parse_time(raw_tid)
                    for r in extracted_rows:
                        if (r['PersonId'] == current_person_id and
                                r['PhaseId'] == current_phase_id and
                                r['SlutTid'] == ''):
                            r['SlutTid'] = s
                current_phase_id = new_phase_id
                current_aspekt   = 0
                s, e = parse_time(raw_tid) if raw_tid is not None else ('', '')
                current_start_tid, current_end_tid = s, e
                if new_phase_id == 'f1':
                    current_global_origin = s
                continue

            has_score = any(pd.notna(row.iloc[i]) for i in range(3, min(6, len(row))))
            if not has_score and col1 == '':
                continue

            if not current_person_id or not current_phase_id:
                continue

            current_aspekt += 1
            extracted_rows.append({
                'PersonId':      current_person_id,
                'RörelseId':     movement_id,
                'PhaseId':       current_phase_id,
                'StartTid':      current_start_tid,
                'SlutTid':       current_end_tid if current_phase_id == 'f3' else '',
                'AspektId':      current_aspekt,
                'Gem_Avvikelse': '' if score is None else (str(round(float(score), 2)) if isinstance(score, (int, float)) else ''),
                'Gem_Säker':     '' if saker is None else (str(round(float(saker), 2)) if isinstance(saker, (int, float)) else ''),
                '_origin': current_global_origin,
            })

        all_extracted_rows.extend(extracted_rows)

    result_df = pd.DataFrame(all_extracted_rows)
    result_df['StartTid'] = result_df.apply(lambda r: to_relative(r['StartTid'], r['_origin']), axis=1)
    result_df['SlutTid']  = result_df.apply(lambda r: to_relative(r['SlutTid'],  r['_origin']), axis=1)
    result_df = result_df.drop(columns=['_origin'])
    return result_df

In [27]:
dataframes = {}

for input_file, output_file in FILES.items():
    ensure_output_directory(output_file)
    df = extract_oma_data(input_file, PHASE_MAPPING, MOVEMENT_MAPPING, PERSON_MAPPING)
    df = df[['PersonId', 'RörelseId', 'PhaseId', 'StartTid', 'SlutTid', 'AspektId', 'Gem_Avvikelse', 'Gem_Säker']]
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    name = os.path.basename(output_file)
    dataframes[name] = df
    print(f"Processing complete. {name} → {len(df)} rows")
    print(df.head(10))
    print(f"\nData saved to: {output_file}\n")

Processing complete. OMA_Score_Kors.csv → 225 rows
  PersonId RörelseId PhaseId  StartTid  SlutTid  AspektId Gem_Avvikelse  \
0      fp1      2a_1      f1       0.0     0.02         1           0.0   
1      fp1      2a_1      f1       0.0     0.02         2           0.0   
2      fp1      2a_1      f1       0.0     0.02         3          0.33   
3      fp1      2a_1      f1       0.0     0.02         4           2.0   
4      fp1      2a_1      f1       0.0     0.02         5                 
5      fp1      2a_1      f1       0.0     0.02         6                 
6      fp1      2a_1      f1       0.0     0.02         7                 
7      fp1      2a_1      f1       0.0     0.02         8           2.0   
8      fp1      2a_1      f1       0.0     0.02         9           2.0   
9      fp1      2a_1      f1       0.0     0.02        10          0.33   

  Gem_Säker  
0      1.33  
1       2.0  
2      1.33  
3       2.0  
4            
5            
6            
7       2.0

In [28]:
print("1. Basic Data Overview")
for name, df in dataframes.items():
    print(f"\n=== {name} ===")
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"\nFirst 5 rows:")
    print(df.head())

1. Basic Data Overview

=== OMA_Score_Kors.csv ===
Shape: 225 rows, 8 columns

First 5 rows:
  PersonId RörelseId PhaseId  StartTid  SlutTid  AspektId Gem_Avvikelse  \
0      fp1      2a_1      f1       0.0     0.02         1           0.0   
1      fp1      2a_1      f1       0.0     0.02         2           0.0   
2      fp1      2a_1      f1       0.0     0.02         3          0.33   
3      fp1      2a_1      f1       0.0     0.02         4           2.0   
4      fp1      2a_1      f1       0.0     0.02         5                 

  Gem_Säker  
0      1.33  
1       2.0  
2      1.33  
3       2.0  
4            

=== OMA_Score_Rak.csv ===
Shape: 225 rows, 8 columns

First 5 rows:
  PersonId RörelseId PhaseId  StartTid  SlutTid  AspektId Gem_Avvikelse  \
0      fp1      2a_1      f1       0.0     0.45         1           0.0   
1      fp1      2a_1      f1       0.0     0.45         2           0.0   
2      fp1      2a_1      f1       0.0     0.45         3           0.0   
3  

In [29]:
print("\n2. Completeness Check")
for name, df in dataframes.items():
    print(f"\n=== {name} ===")
    print(f"Unique persons: {sorted(df['PersonId'].unique())}")
    print(f"Unique movements: {df['RörelseId'].value_counts().to_dict()}")
    print(f"Unique phases: {df['PhaseId'].value_counts().to_dict()}")
    print(f"Total aspects (rows): {len(df)}")


2. Completeness Check

=== OMA_Score_Kors.csv ===
Unique persons: ['fp1', 'fp2', 'fp3', 'fp4', 'fp5']
Unique movements: {'2a_2': 115, '2a_1': 110}
Unique phases: {'f1': 130, 'f3': 85, 'f2': 10}
Total aspects (rows): 225

=== OMA_Score_Rak.csv ===
Unique persons: ['fp1', 'fp2', 'fp3', 'fp4', 'fp5']
Unique movements: {'2a_2': 115, '2a_1': 110}
Unique phases: {'f1': 130, 'f3': 85, 'f2': 10}
Total aspects (rows): 225


In [30]:
print("\n3. Mapping Validation")
for name, df in dataframes.items():
    print(f"\n=== {name} ===")
    print(f"Phase IDs found: {sorted(df['PhaseId'].unique())}")
    print(f"Movement IDs found: {sorted(df['RörelseId'].unique())}")


3. Mapping Validation

=== OMA_Score_Kors.csv ===
Phase IDs found: ['f1', 'f2', 'f3']
Movement IDs found: ['2a_1', '2a_2']

=== OMA_Score_Rak.csv ===
Phase IDs found: ['f1', 'f2', 'f3']
Movement IDs found: ['2a_1', '2a_2']


In [31]:
print("\n4. Time Consistency Checks")
for name, df in dataframes.items():
    print(f"\n=== {name} ===")
    df_num = df.copy()
    df_num['StartTid_num'] = pd.to_numeric(df_num['StartTid'], errors='coerce')
    df_num['SlutTid_num']  = pd.to_numeric(df_num['SlutTid'],  errors='coerce')
    invalid = df_num[(df_num['StartTid_num'] > df_num['SlutTid_num']) & df_num['SlutTid_num'].notna()]
    if not invalid.empty:
        print("Found rows with invalid time ranges (start > end):")
        print(invalid[['PersonId', 'PhaseId', 'StartTid', 'SlutTid']])
    else:
        print("All time ranges are valid (start ≤ end)")


4. Time Consistency Checks

=== OMA_Score_Kors.csv ===
All time ranges are valid (start ≤ end)

=== OMA_Score_Rak.csv ===
All time ranges are valid (start ≤ end)


In [32]:
print("\n5. Data Integrity and Scores")
for name, df in dataframes.items():
    print(f"\n=== {name} ===")
    missing = df[['PersonId', 'RörelseId', 'PhaseId', 'AspektId', 'Gem_Avvikelse']].isnull().sum()
    print("Missing values:")
    for col, count in missing.items():
        print(f"  {col}: {count}")
    print("\nScore distribution:")
    print(df['Gem_Avvikelse'].astype(str).value_counts().sort_index())


5. Data Integrity and Scores

=== OMA_Score_Kors.csv ===
Missing values:
  PersonId: 0
  RörelseId: 0
  PhaseId: 0
  AspektId: 0
  Gem_Avvikelse: 0

Score distribution:
Gem_Avvikelse
        78
0.0     22
0.33    33
0.67    18
1.0     12
1.33    14
1.5      1
1.67    10
2.0     37
Name: count, dtype: int64

=== OMA_Score_Rak.csv ===
Missing values:
  PersonId: 0
  RörelseId: 0
  PhaseId: 0
  AspektId: 0
  Gem_Avvikelse: 0

Score distribution:
Gem_Avvikelse
       51
0.0    75
1.0    84
2.0    15
Name: count, dtype: int64


In [33]:
print("\n6. Data Structure Example")
df_kors = dataframes['OMA_Score_Kors.csv']
example_person = df_kors['PersonId'].iloc[0]
print(f"Showing data structure for: {example_person}\n")

person_data = df_kors[df_kors['PersonId'] == example_person]
for movement, movement_group in person_data.groupby('RörelseId'):
    print(f"Movement: {movement}")
    for phase, phase_group in movement_group.groupby('PhaseId'):
        print(f"  Phase: {phase}")
        aspects = phase_group[['AspektId', 'StartTid', 'SlutTid', 'Gem_Avvikelse']].sort_values('AspektId')
        print(aspects.to_string(index=False))
    print("-" * 20)


6. Data Structure Example
Showing data structure for: fp1

Movement: 2a_1
  Phase: f1
 AspektId  StartTid  SlutTid Gem_Avvikelse
        1       0.0     0.02           0.0
        2       0.0     0.02           0.0
        3       0.0     0.02          0.33
        4       0.0     0.02           2.0
        5       0.0     0.02              
        6       0.0     0.02              
        7       0.0     0.02              
        8       0.0     0.02           2.0
        9       0.0     0.02           2.0
       10       0.0     0.02          0.33
       11       0.0     0.02           0.0
  Phase: f2
 AspektId  StartTid  SlutTid Gem_Avvikelse
        1      0.02     0.02          1.67
  Phase: f3
 AspektId  StartTid  SlutTid Gem_Avvikelse
        1      0.02     1.87          0.33
        2      0.02     1.87          0.33
        3      0.02     1.87           1.0
        4      0.02     1.87           2.0
        5      0.02     1.87          0.33
        6      0.02     1.87 